# 052_pipeline_run_report
Read-only views of the run log: the latest batch flow by flow, the slowest flows, and any
failures with their messages.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_pipeline_run_latest AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY event_time DESC) AS event_rank
    FROM {CTRL_DATABASE}.pipeline_run_log
)
WHERE event_rank = 1
""")

latest_batch = spark.sql("SELECT MAX(batch_id) AS b FROM v_pipeline_run_latest WHERE batch_id LIKE 'batch_%'").collect()[0]["b"]
print(f"Latest batch: {latest_batch}")

## Latest batch, flow by flow

In [ ]:
spark.sql(f"""
SELECT
    layer,
    control_id,
    pipeline_name,
    status,
    retry_attempt,
    duration_seconds,
    rows_written
FROM v_pipeline_run_latest
WHERE batch_id = '{latest_batch}'
ORDER BY
    CASE layer WHEN 'BRONZE' THEN 1 WHEN 'SILVER' THEN 2 WHEN 'GOLD' THEN 3 ELSE 4 END,
    control_id
""").show(100, truncate=False)

## Failures in the latest batch

In [ ]:
spark.sql(f"""
SELECT pipeline_name, retry_attempt, error_message
FROM v_pipeline_run_latest
WHERE batch_id = '{latest_batch}'
  AND status IN ('FAILED', 'SKIPPED')
""").show(truncate=False)